### 面向离子阱后端编译示例

QLLVM编译器支持编译生存面向H-series离子阱后端的量子程序，下面是编译运行示例

1、基于QASM模拟器运行编译后程序

In [10]:
import os
from pytket.qasm import circuit_from_qasm
from qiskit_aer import AerSimulator
from pytket.extensions.qiskit import tk_to_qiskit 
from qiskit import QuantumCircuit
from qiskit.quantum_info.analysis import hellinger_fidelity

qasm_file = "../examples/qasm/bell.qasm"
with open(qasm_file,'r')as file:
        qasm_str = file.read()
print("========================origin qasm========================")
print(qasm_str)

algorithm_name, _ = os.path.splitext(os.path.basename(qasm_file))
compile_command = f"qllvm {qasm_file} -qrt nisq -qpu trappedion-qasm -o ./{algorithm_name} -O1"
os.system(compile_command)

comiled_qasm = f"./{algorithm_name}.qasm"
with open(comiled_qasm,'r')as file:
        qasm_str = file.read()
print("========================compiled qasm========================")
print(qasm_str)

circ1 = circuit_from_qasm(comiled_qasm)
qc1 = tk_to_qiskit(circ1)
qc2 = QuantumCircuit().from_qasm_file(qasm_file)
backend = AerSimulator()
job1 = backend.run(qc1, shots=1000).result()
job2 = backend.run(qc2, shots=1000).result()
print(f"编译前线路测量结果：{job1.get_counts()}")
print(f"编译前后路测量结果：{job2.get_counts()}")
result = hellinger_fidelity(job1.get_counts(), job2.get_counts())
print(result)

========================origin qasm========================
OPENQASM 2.0;
include "qelib1.inc";

qreg q[2];
creg c[2];

h q[0];
cx q[0], q[1];

measure q[0] -> c[0];
measure q[1] -> c[1];
========================compiled qasm========================
OPENQASM 2.0;
include "hqslib1.inc";
qreg q[2];
creg c[2];
rz(1*pi) q[0];
U1q (0.5*pi, 0.5*pi) q[0];
U1q (-0.5*pi,0.5*pi) q[1];
rzz (0.5*pi) q[0], q[1];
rz(-0.5*pi) q[0];
U1q (0.5*pi, pi) q[1];
rz(-0.5*pi) q[1];
measure q[0] -> c[0];
measure q[1] -> c[1];

编译前线路测量结果：{'00': 493, '11': 507}
编译前后路测量结果：{'11': 489, '00': 511}
0.9996759948142369


2、基于本地离子阱模拟器运行编译后程序

In [ ]:
import dataclasses
from pytket import Circuit
from pytket.extensions.quantinuum import QuantinuumBackend
from pytket.extensions.quantinuum.backends.data import H2

import dataclasses
from pytket.circuit import OpType
from pytket.extensions.quantinuum.backends.data import H2
from pytket.qasm import circuit_from_qasm
import os

local_h2 = dataclasses.replace(
    H2,
    local_emulator=True,
    gate_set=frozenset(
        H2.gate_set
        | {
            OpType.Measure,
            OpType.Reset,
            OpType.Barrier,
        }
    ),
)
backend = QuantinuumBackend("H2-1LE", data=local_h2)

qasm_file = "../examples/qasm/bell.qasm"
with open(qasm_file,'r')as file:
        qasm_str = file.read()
print("========================origin qasm========================")
print(qasm_str)

algorithm_name, _ = os.path.splitext(os.path.basename(qasm_file))
compile_command = f"qllvm {qasm_file} -qrt nisq -qpu trappedion-qasm -o ./{algorithm_name} -O1"
os.system(compile_command)

comiled_qasm = f"./{algorithm_name}.qasm"
with open(comiled_qasm,'r')as file:
        qasm_str = file.read()
print("========================compiled qasm========================")
print(qasm_str)

circ = circuit_from_qasm(qasm_file)
qc1 = backend.get_compiled_circuit(circ)
circ2 = circuit_from_qasm(comiled_qasm)
handles1 = backend.process_circuits(
    [qc1],
    n_shots=1000,
    noisy_simulation=False,
    seed=12345,
)
handles2 = backend.process_circuits(
    [circ2],
    n_shots=1000,
    noisy_simulation=False,
    seed=12345,
)

result1 = backend.get_result(handles1[0])
result2 = backend.get_result(handles2[0])
print(f"编译前线路测量结果：{result1.get_counts()}")
print(f"编译后线路测量结果：{result2.get_counts()}")

========================origin qasm========================
OPENQASM 2.0;
include "qelib1.inc";

qreg q[2];
creg c[2];

h q[0];
cx q[0], q[1];

measure q[0] -> c[0];
measure q[1] -> c[1];
========================compiled qasm========================
OPENQASM 2.0;
include "hqslib1.inc";
qreg q[2];
creg c[2];
rz(1*pi) q[0];
U1q (0.5*pi, 0.5*pi) q[0];
U1q (-0.5*pi,0.5*pi) q[1];
rzz (0.5*pi) q[0], q[1];
rz(-0.5*pi) q[0];
U1q (0.5*pi, pi) q[1];
rz(-0.5*pi) q[1];
measure q[0] -> c[0];
measure q[1] -> c[1];

编译前线路测量结果：Counter({(0, 0): 500, (1, 1): 500})
编译后线路测量结果：Counter({(0, 0): 512, (1, 1): 488})
